# Feature Engineering и временной split
Подготовка перед обучением модели


## 1. Загрузка данных


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

df = pd.read_parquet(
    "../data/interim/hotel_bookings_clean.parquet"
)

df.shape


(119390, 32)

## 2. Создаём дату заезда


In [2]:
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str)
    + '-'
    + df['arrival_date_month'].astype(str)
    + '-'
    + df['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)

In [3]:
df["arrival_date"].agg(["min", "max"])

min   2015-07-01
max   2017-08-31
Name: arrival_date, dtype: datetime64[us]

## 3. Делаем split


Наша задача - предсказывать отмену в момент создания бронирования, поэтому делить будем не по arrival date, а по моменту, когда бронь появилась. 

Для этого создадим новый признак `booking_date`: `arrival_date` - `lead_time`


In [4]:
df['booking_date'] = (
    df['arrival_date']
    - pd.to_timedelta(df['lead_time'], unit='D')
)

In [5]:
df[["arrival_date", "lead_time", "booking_date"]].head()

,arrival_date,lead_time,booking_date
0,2015-07-01,342,2014-07-24
1,2015-07-01,737,2013-06-24
2,2015-07-01,7,2015-06-24
3,2015-07-01,13,2015-06-18
4,2015-07-01,14,2015-06-17


In [6]:
df["booking_date"].agg(["min", "max"])

min   2013-06-24
max   2017-08-31
Name: booking_date, dtype: datetime64[us]

## 4. Feature Engineering

Создаём новые признаки на основе выводов из предыдущих ноутбуков.


### Количество гостей - `total_guests`


In [7]:
df["total_guests"] = (
    df["adults"]
    + df["children"].fillna(0)
    + df["babies"]
)

### Количество ночей - `total_nights`


In [8]:
df["total_nights"] = (
    df["stays_in_week_nights"]
    + df["stays_in_weekend_nights"]
)

### Была ли раньше отмена - `has_previous_cancellation`
Из EDA


In [9]:
df["has_previous_cancellation"] = (
    df["previous_cancellations"] > 0
).astype(int)

### ADR = 0 - `is_zero_adr`

Брони с `ADR = 0` заметно отличаются по доле отмен. Причину по этим данным определить нельзя, поэтому пока оставляю отдельный флаг `is_zero_adr` и позже проверю, помогает ли он модели.


In [10]:
df["is_zero_adr"] = (df["adr"] == 0).astype(int)

## 5. Временной split


Посмотрим на распределение по новому признаку `booking_date`


In [11]:
df.groupby(
    df["booking_date"].dt.to_period("M")
).size()

booking_date
2013-06       1
2014-03       1
2014-04       4
2014-06       2
2014-07       3
2014-08       5
2014-09      18
2014-10    2535
2014-11      68
2015-01    1165
2015-02     799
2015-03     882
2015-04     942
2015-05     971
2015-06    1382
2015-07    5196
2015-08    3281
2015-09    3740
2015-10    4878
2015-11    4254
2015-12    4155
2016-01    7654
2016-02    6897
2016-03    5429
2016-04    4940
2016-05    3999
2016-06    3052
2016-07    3279
2016-08    3954
2016-09    4427
2016-10    4716
2016-11    5183
2016-12    5013
2017-01    7869
2017-02    5772
2017-03    3609
2017-04    2736
2017-05    2883
2017-06    1626
2017-07    1340
2017-08     730
Freq: M, dtype: int64

По распределению, я бы раскидал объекты так:

* `TRAIN`       всё до 2016-11-01
* `VALIDATION`  2016-11-01 — 2017-01-31
* `TEST`        с 2017-02-01

Получается примерно 70 / 15 / 15, при этом временной порядок сохраняется.


In [12]:
train = df[
    df["booking_date"] < "2016-11-01"
].copy()

val = df[
    (df["booking_date"] >= "2016-11-01")
    & (df["booking_date"] < "2017-02-01")
].copy()

test = df[
    df["booking_date"] >= "2017-02-01"
].copy()

Проверим размеры выборок, даты и долю отмен:


In [13]:
splits = {
    "train": train,
    "validation": val,
    "test": test
}

summary = []

for name, data in splits.items():
    summary.append({
        "split": name,
        "rows": len(data),
        "share": len(data) / len(df),
        "date_from": data["booking_date"].min(),
        "date_to": data["booking_date"].max(),
        "cancellation_rate": data["is_canceled"].mean()
    })

split_summary = pd.DataFrame(summary)

split_summary

,split,rows,share,date_from,date_to,cancellation_rate
0,train,82629,0.692093,2013-06-24,2016-10-31,0.380109
1,validation,18065,0.151311,2016-11-01,2017-01-31,0.392804
2,test,18696,0.156596,2017-02-01,2017-08-31,0.305948


In [15]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)

train.to_parquet("../data/processed/train.parquet", index=False)
val.to_parquet("../data/processed/validation.parquet", index=False)
test.to_parquet("../data/processed/test.parquet", index=False)


## Вывод
Доля отмен на `test` заметно ниже, чем на `train` и `validation`: разница с validation почти 9 процентных пунктов. Это хороший сигнал, что будущий период отличается от данных обучения. На таком split будет видно, насколько модель переносится на новые данные.
